# Tiền xử lý và ánh xạ nhãn

Chạy notebook EDA trước. Các ví dụ lấy từ train. CNN và MobileNetV2 nhận tensor RGB thô, chuẩn hóa bằng lớp tích hợp trong checkpoint. Augmentation chỉ áp dụng lúc huấn luyện.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image as NotebookImage, display

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "src" / "config.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Chuyển thư mục làm việc về dự án trước. Trên Colab, xem hướng dẫn trong README.")
sys.path.insert(0, str(project_root))

from src.config import Paths
paths = Paths(project_root)

In [ ]:
import pandas as pd
import tensorflow as tf

from src.config import CNN_SIZE, MOBILENET_SIZE
from src.dataset import load_manifest
from src.hog_features import extract_hog
from src.preprocessing import image_dataset, read_image

manifest = load_manifest(paths)
display(pd.Series(manifest.class_to_index, name="class_index").to_frame())
train_paths, train_labels = manifest.for_split(paths, "train")
image = read_image(train_paths[0])
display(image.copy().resize((256, 256)))
print("HOG", extract_hog(image).shape)

In [ ]:
for name, size in [("CNN", CNN_SIZE), ("MobileNetV2", MOBILENET_SIZE)]:
    batch, labels = next(iter(image_dataset(train_paths[:4], train_labels[:4], size, 4, training=False)))
    normalized = batch / 255 if name == "CNN" else tf.keras.applications.mobilenet_v2.preprocess_input(batch)
    print(name, normalized.shape, "min", float(tf.reduce_min(normalized)), "max", float(tf.reduce_max(normalized)))
    print("Nhãn", labels.numpy())